Решал похожую задачу летом (в том решении есть ошибки), но, так как здесь очень мало изображений и решается немного другая задача, оставил только более-менее значимые признаки.

https://colab.research.google.com/drive/1D4JlqYb0l_hqhe4XxcG9wIvN9lXsNImJ?usp=sharing

# Извлечение признаков

In [42]:
import os
from glob import glob
from typing import Union, List
from pathlib import Path

import numpy as np
import pandas as pd

import cv2
from skimage.feature import graycomatrix, graycoprops, local_binary_pattern
from skimage.color import rgb2gray, rgb2hsv, rgb2lab

from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier

In [43]:
import warnings
warnings.filterwarnings("ignore", message="X does not have valid feature names")

In [44]:
RESIZE_TO = (512, 512)
LBP_RADIUS = 3
LBP_POINTS = 8 * LBP_RADIUS
GLCM_DISTANCES = [1]
GLCM_ANGLES = [0, np.pi/4, np.pi/2, 3*np.pi/4]

def extract_features(image_path: Union[str, Path]) -> List[float]:
    """Извлекает признаки из изображения"""
    # Загрузка и предобработка
    image = cv2.imread(image_path)
    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    image = cv2.resize(image, RESIZE_TO)
    blurred = cv2.GaussianBlur(image, (5, 5), 0)

    gray = rgb2gray(blurred)
    hsv = rgb2hsv(blurred)
    lab = rgb2lab(blurred)

    features = []

    ### 1. ЦВЕТОВЫЕ ПРИЗНАКИ
    # RGB (без канала B)
    red_mean = np.mean(blurred[:, :, 0])
    green_mean = np.mean(blurred[:, :, 1])
    features.extend([red_mean, green_mean])

    # Отношение зелёного канала к красному
    r_g_ratio = red_mean / (green_mean + 1e-7)
    features.append(r_g_ratio)

    # Разность зелёного и красного каналов
    r_minus_g = red_mean - green_mean
    features.append(r_minus_g)

    # HSV
    h_mean = np.mean(hsv[:, :, 0])  
    s_mean = np.mean(hsv[:, :, 1])  
    v_mean = np.mean(hsv[:, :, 2])  
    features.extend([h_mean, s_mean, v_mean])

    # LAB (без канала B)
    l_mean = np.mean(lab[:, :, 0])  
    a_mean = np.mean(lab[:, :, 1])
    features.extend([l_mean, a_mean])
    
    # Стандартные отклонения для каналов Hue и Red
    features.append(np.std(hsv[:, :, 0]))  
    features.append(np.std(blurred[:, :, 0])) 
    
    ### 3. ТЕКСТУРА (GLCM)
    glcm = graycomatrix(
        (gray * 255).astype(np.uint8),
        distances=GLCM_DISTANCES,
        angles=GLCM_ANGLES,
        levels=256,
        symmetric=True
    )
    
    # Только contrast и homogeneity 
    contrast = graycoprops(glcm, "contrast").ravel()
    homogeneity = graycoprops(glcm, "homogeneity").ravel()
    
    features.extend(contrast)
    features.extend(homogeneity)

    ### 4. LBP
    
    lbp = local_binary_pattern((gray * 255).astype(np.uint8), P=LBP_POINTS, R=LBP_RADIUS, method="uniform")

    hist, _ = np.histogram(lbp, bins=10, range=(0, LBP_POINTS + 2))
    hist = hist.astype("float") / (hist.sum() + 1e-7)
    features.extend(hist)

    ### 5. ГИСТОГРАММЫ
    
    # Гистограмма Hue 
    hsv_cv = cv2.cvtColor(blurred, cv2.COLOR_RGB2HSV)
    hue_hist = cv2.calcHist([hsv_cv], [0], None, [8], [0, 180]).flatten()
    hue_hist = hue_hist / (hue_hist.sum() + 1e-7)
    features.extend(hue_hist)
    
    # Гистограмма Red 
    red_hist = cv2.calcHist([blurred], [0], None, [8], [0, 256]).flatten()
    red_hist = red_hist / (red_hist.sum() + 1e-7)
    features.extend(red_hist)
    
    # Гистограмма Saturation
    sat_hist = cv2.calcHist([hsv_cv], [1], None, [8], [0, 256]).flatten()
    sat_hist = sat_hist / (sat_hist.sum() + 1e-7)
    features.extend(sat_hist)
    
    return features

In [45]:
dataset_path = "./train"
classes = ["fully_ripened", "half_ripened", "green"]

class_to_label = {cls: i for i, cls in enumerate(classes)}

data = []
labels = []

for class_name in classes:
    image_paths = []
    for ext in ["*.jpg"]:
        image_paths.extend(glob(os.path.join(dataset_path, class_name, ext)))

    for path in image_paths:
        features = extract_features(path)
        if features is not None:
            data.append(features)
            labels.append(class_to_label[class_name])

columns = [
    "R_mean", "G_mean", "R_G_ratio", "R_minus_G",
    "H_mean", "S_mean", "V_mean", 
    "L_mean", "A_mean",
    "H_std", "R_std",
    *[f"GLCM_contrast_{i}" for i in range(4)],
    *[f"GLCM_homogeneity_{i}" for i in range(4)],
    *[f"LBP_{i}" for i in range(10)],
    *[f"Hist_H_{i}" for i in range(8)],
    *[f"Hist_R_{i}" for i in range(8)],
    *[f"Hist_S_{i}" for i in range(8)]
]

df = pd.DataFrame(data, columns=columns)
df["label"] = labels
df.to_csv("tomato_features_train.csv", index=False)

# Обучение

In [46]:
# Чтение файла
df = pd.read_csv("tomato_features_train.csv")

# Разделение на признаки и целевую переменную
X = df.drop("label", axis=1)
y = df["label"]

# Обучение Random Forest
rf_classifier = RandomForestClassifier(n_estimators=100, random_state=2025)
rf_classifier.fit(X, y)

# Обучение KNN
knn_classifier = KNeighborsClassifier(n_neighbors=3)
knn_classifier.fit(X, y)

,n_neighbors,3
,weights,'uniform'
,algorithm,'auto'
,leaf_size,30
,p,2
,metric,'minkowski'
,metric_params,None
,n_jobs,None


In [47]:
feature_importance = rf_classifier.feature_importances_

indices = np.argsort(feature_importance)[::-1]

print("Важность признаков:")
for i, idx in enumerate(indices):
    print(f"{i+1}. {columns[idx]}: {feature_importance[idx]:.4f}")

Важность признаков:
1. R_G_ratio: 0.1460
2. A_mean: 0.1136
3. R_minus_G: 0.1064
4. Hist_H_1: 0.0941
5. Hist_H_0: 0.0720
6. H_mean: 0.0719
7. G_mean: 0.0575
8. L_mean: 0.0309
9. H_std: 0.0220
10. Hist_R_7: 0.0203
11. Hist_H_6: 0.0202
12. Hist_R_5: 0.0182
13. Hist_R_0: 0.0171
14. Hist_H_5: 0.0163
15. Hist_S_6: 0.0148
16. V_mean: 0.0134
17. S_mean: 0.0118
18. LBP_2: 0.0107
19. Hist_H_4: 0.0096
20. Hist_S_7: 0.0094
21. LBP_6: 0.0093
22. Hist_S_2: 0.0086
23. LBP_9: 0.0081
24. Hist_H_2: 0.0078
25. GLCM_homogeneity_2: 0.0075
26. Hist_S_0: 0.0062
27. Hist_R_6: 0.0057
28. Hist_H_3: 0.0056
29. R_mean: 0.0051
30. Hist_S_3: 0.0050
31. Hist_R_1: 0.0049
32. GLCM_contrast_0: 0.0048
33. Hist_R_4: 0.0042
34. R_std: 0.0037
35. LBP_5: 0.0036
36. Hist_R_3: 0.0035
37. GLCM_contrast_2: 0.0035
38. Hist_S_5: 0.0031
39. Hist_H_7: 0.0030
40. Hist_S_4: 0.0029
41. GLCM_homogeneity_0: 0.0029
42. LBP_0: 0.0026
43. Hist_S_1: 0.0025
44. LBP_1: 0.0024
45. Hist_R_2: 0.0024
46. LBP_3: 0.0018
47. GLCM_contrast_1: 0.0017


# Классификация

In [48]:
dataset_path = "./test"

In [49]:
image_paths = []
for ext in ["*.jpg"]:
    image_paths.extend(glob(os.path.join(dataset_path, ext)))

In [50]:
def classify_tomato(image_path: Union[str, Path], model) -> int:
    "Классифицирует изображение томата с помощью заданной модели"
    features = extract_features(image_path)
    
    y_proba = model.predict_proba([features])[0]
    predicted_class = np.argmax(y_proba)

    return int(predicted_class) + 1

In [51]:
with open("rf_answer.txt", "w") as file:
    for i, image_path in enumerate(image_paths):
        tomato_class = classify_tomato(image_path, rf_classifier)
        if i == len(image_paths) - 1:
            file.write(str(tomato_class))
        else:
            file.write(str(tomato_class) + "\n")

In [52]:
with open("knn_answer.txt", "w") as file:
    for i, image_path in enumerate(image_paths):
        tomato_class = classify_tomato(image_path, knn_classifier)
        if i == len(image_paths) - 1:
            file.write(str(tomato_class))
        else:
            file.write(str(tomato_class) + "\n")

In [53]:
with open("rf_answer.txt") as f1, open("knn_answer.txt") as f2:
    for i, (line1, line2) in enumerate(zip(f1, f2)):
        if line1.strip() != line2.strip():
            print(f"Строка {i+1}: {line1.strip()} != {line2.strip()}")

Строка 35: 1 != 2
Строка 98: 1 != 2


In [54]:
with open("answer.txt", "w") as file:
    for i, image_path in enumerate(image_paths):
        tomato_class = classify_tomato(image_path, rf_classifier)
        if i == len(image_paths) - 1:
            file.write(str(tomato_class))
        else:
            file.write(str(tomato_class) + "\n")

In [56]:
with open("answer_resized.txt") as f1, open("answer_without_resize.txt") as f2:
    for i, (line1, line2) in enumerate(zip(f1, f2)):
        if line1.strip() != line2.strip():
            print(f"Строка {i+1}: {line1.strip()} != {line2.strip()}")